# TerraDelta offline prediction

Reads AIF_INPUT_DIR and writes AIF_PREDICTION_PATH. No training, network calls or submit commands.


In [ ]:
# Offline inference. All assets are bundled in this ZIP.
import os
import sys
from pathlib import Path

# Locate bundled resources using execution/package anchors, never a CWD assumption.
def _submission_root():
    anchors = []
    for name in ("__file__", "__vsc_ipynb_file__", "__notebook_path__"):
        value = globals().get(name)
        if value:
            anchors.append(Path(value).resolve().parent)
    for name in ("AIF_SUBMISSION_DIR", "AIF_NOTEBOOK_PATH"):
        value = os.environ.get(name)
        if value:
            path = Path(value).resolve()
            anchors.append(path.parent if path.suffix in {".ipynb", ".py"} else path)
    for value in sys.argv:
        if value.endswith((".ipynb", ".py")) and Path(value).is_file():
            anchors.append(Path(value).resolve().parent)
    module = sys.modules.get("terradelta")
    if module is not None and getattr(module, "__file__", None):
        anchors.append(Path(module.__file__).resolve().parent)
    anchors.extend(Path(value).resolve() for value in sys.path[:32] if value)
    # The official runner extracts here; root execution remains supported too.
    anchors.extend((Path("/aif/submission"), Path.cwd()))
    checked = []
    for anchor in anchors:
        for candidate in (anchor, *list(anchor.parents)[:4]):
            if candidate in checked:
                continue
            checked.append(candidate)
            required = ("assets/config.yaml", "assets/model/model.pt",
                        "assets/code/terradelta/__init__.py")
            if all((candidate / name).is_file() for name in required):
                return candidate
    raise FileNotFoundError("Submission assets not found; supply the notebook path "
                            "or AIF_SUBMISSION_DIR. Checked: " +
                            ", ".join(map(str, checked)))

ROOT = _submission_root()
sys.path.insert(0, str(ROOT / "assets/code"))
import torch
from terradelta.utils.io import load_config
from terradelta.inference.predictor import predict_directory

CONFIG = load_config(ROOT / "assets/config.yaml")
INPUT_DIR = Path(os.environ.get("AIF_INPUT_DIR", "./input"))
PREDICTION_PATH = Path(os.environ.get("AIF_PREDICTION_PATH", "./prediction.csv"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# CPU fallback is limited to accelerator readiness, before processing input.
from terradelta.models.factory import build_model
from terradelta.models.checkpoint import load_checkpoint
if DEVICE == "cuda":
    try:
        probe = build_model({"encoder_weights": None}).to(DEVICE).eval()
        load_checkpoint(ROOT / "assets/model/model.pt", probe)
        with torch.inference_mode():
            probe(torch.zeros(1, 6, 256, 256, device=DEVICE))
        del probe
    except (RuntimeError, torch.cuda.OutOfMemoryError) as error:
        print("CUDA readiness failed; using CPU:", type(error).__name__)
        DEVICE = "cpu"
    torch.cuda.empty_cache()
rows = predict_directory(INPUT_DIR, PREDICTION_PATH,
    ROOT / "assets/model/model.pt", CONFIG, device=DEVICE)
print("Saved", len(rows), "pairs to", PREDICTION_PATH)
